In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


train_dir = 'train'
validation_dir = 'validation'
test_dir = 'test'


train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)


validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)


train_generator = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    directory=validation_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    directory=test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


class_labels = list(train_generator.class_indices.keys())
class_labels

In [ ]:
len(train_generator)

In [ ]:
train_generator.samples

In [ ]:
train_generator.num_classes

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout


base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


base_model.trainable = False


x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)


extract_feat_model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
extract_feat_model.summary()

In [ ]:
extract_feat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_extract = extract_feat_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=100,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    verbose=1
)

In [ ]:
fine_tuned_model = extract_feat_model

base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

fine_tuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_finetune = fine_tuned_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(history, model_name="Model"):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
    plt.title(f'{model_name} - Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_accuracy(history_extract, "Extract Features Model")

In [ ]:
def plot_loss(history, model_name="Model"):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['loss'], label='Training Loss', marker='o')
    plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
    plt.title(f'{model_name} - Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_loss(history_finetune, "Fine-Tuned Model")

In [ ]:
plot_accuracy(history_finetune, "Fine-Tuned Model")

In [ ]:
import numpy as np

def plot_test_image_with_prediction(generator, model, index_to_plot, class_labels, model_name=""):
    
    generator.reset()
    
    batch_idx = index_to_plot // generator.batch_size
    img_index = index_to_plot % generator.batch_size
    
    images, true_labels = generator[batch_idx]
    
    image = images[img_index]
    true_label_idx = np.argmax(true_labels[img_index])
    
    prediction = model.predict(np.expand_dims(image, axis=0), verbose=0)
    pred_label_idx = np.argmax(prediction[0])
    
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f'{model_name}\nTrue: {class_labels[true_label_idx]}\nPredicted: {class_labels[pred_label_idx]}')
    plt.axis('off')
    plt.show()
    
    return true_label_idx, pred_label_idx

plot_test_image_with_prediction(
    test_generator, 
    extract_feat_model, 
    index_to_plot=1, 
    class_labels=class_labels,
    model_name="Extract Features Model"
)

In [ ]:
plot_test_image_with_prediction(
    test_generator, 
    fine_tuned_model, 
    index_to_plot=1, 
    class_labels=class_labels,
    model_name="Fine-Tuned Model"
)

In [ ]:
extract_loss, extract_acc = extract_feat_model.evaluate(test_generator, verbose=0)

In [ ]:
extract_loss

In [ ]:
extract_acc

In [ ]:
fine_loss, fine_acc = fine_tuned_model.evaluate(test_generator, verbose=0)

In [ ]:
fine_loss

In [ ]:
fine_acc